# Symbiota API — How It Works

**Symbiota** is the open-source software that powers biodiversity portals like [MyCoPortal](https://mycoportal.org) (fungi), BryophytePortal (mosses/liverworts), and many others.

This notebook explores the three endpoints our `MyCoPortal.py` client uses, plus a fourth (`occurrence/search`) that we don't use yet but is valuable:

| # | Endpoint | What it does |
|---|---|---|
| 1 | `gettaxasuggest` RPC | Name autocomplete — converts a name string to a taxon ID (`tid`) |
| 2 | `/api/v2/taxonomy/{tid}` | Taxonomy record — accepted/synonym status, authorship, classification |
| 3 | `taxa/index.php?tid={tid}` | HTML page — scrape the synonym list (not available via REST) |
| 4 | `/api/v2/occurrence/search` | Specimen records — where and when a species was collected |

In [1]:
import re
import requests

BASE_URL = "https://mycoportal.org/portal"
HEADERS = {"User-Agent": "Mozilla/5.0"}

---
## 1. `gettaxasuggest` — Name autocomplete

This is an RPC endpoint (not REST) that works like a search-as-you-type autocomplete.  
You pass a partial or full name via `?term=` and get back a list of matching taxon records.

Each result has two fields:
- **`id`** — the taxon ID (`tid`), the primary key used by every other endpoint
- **`label`** — the full name with authorship, e.g. `"Gymnopus dryophilus (Bull.) Murrill"`

**Gotcha:** the portal stores names with a capitalised genus — a lowercase query like `"gymnopus"` returns no results.

In [2]:
# Partial name — returns all matching taxa (species, varieties, etc.)
resp = requests.get(
    f"{BASE_URL}/taxa/taxonomy/rpc/gettaxasuggest.php",
    params={"term": "Gymnopus dry"},
    headers=HEADERS,
)
resp.json()

[{'id': '183886', 'label': 'Gymnopus dryophilus (Bull.) Murrill'},
 {'id': '500556',
  'label': 'Gymnopus dryophilus var. dryophilus (Bull.) Murrill'},
 {'id': '500557',
  'label': 'Gymnopus dryophilus var. lanipes (Malençon & Bertault) A. Ortega, Antonín & Esteve-Rav.'}]

The three hits are the accepted species plus two infraspecific varieties (var.).  
In our script we only want an exact species-level match, so we filter with a regex on `label`.

In [3]:
# Exact match — filter by whether the label starts with the full species name
species_name = "Gymnopus dryophilus"
resp = requests.get(
    f"{BASE_URL}/taxa/taxonomy/rpc/gettaxasuggest.php",
    params={"term": species_name},
    headers=HEADERS,
)

tid = None
for item in resp.json():
    if re.match(rf"^{re.escape(species_name)}(\s|$)", item["label"]):
        tid = int(item["id"])
        print(f"Found: label={item['label']!r}  tid={tid}")
        break

Found: label='Gymnopus dryophilus (Bull.) Murrill'  tid=183886


---
## 2. `/api/v2/taxonomy/{tid}` — Taxonomy record

Given a `tid`, this REST endpoint returns the full taxonomy record.  
The most important field for our use case is **`status`**, which is either `"accepted"` or `"synonym"`.

### 2a. Accepted taxon

When `status` is `"accepted"`, there is no `accepted` block — the taxon *is* the accepted name.

In [4]:
# tid 183886 = Gymnopus dryophilus (the accepted name)
resp = requests.get(f"{BASE_URL}/api/v2/taxonomy/183886", headers=HEADERS)
data = resp.json()

print("status          :", data["status"])
print("scientificName  :", data["scientificName"])
print("author          :", data["author"])
print("rankID          :", data["rankID"], "  (220 = species)")
print("'accepted' key? :", "accepted" in data)

status          : accepted
scientificName  : Gymnopus dryophilus
author          : (Bull.) Murrill
rankID          : 220   (220 = species)
'accepted' key? : False


In [5]:
# The classification array gives the full taxonomic path from kingdom down to genus
for rank in data["classification"]:
    print(f"  rankid={rank['rankid']:4d}  {rank['scientificName']}")

  rankid=  10  Fungi
  rankid=  30  Basidiomycota
  rankid=  40  Agaricomycotina
  rankid=  60  Agaricomycetes
  rankid=  70  Agaricomycetidae
  rankid= 100  Agaricales
  rankid= 100  Boletales
  rankid= 140  Marasmiaceae
  rankid= 140  Suillaceae
  rankid= 180  Gymnopus


### 2b. Synonym taxon

When `status` is `"synonym"`, the response includes an **`accepted`** block that points to the accepted taxon.  
This is how we resolve a synonym to its accepted name without scraping HTML.

In [6]:
# tid 88869 = Agaricus dryophilus (a synonym)
resp = requests.get(f"{BASE_URL}/api/v2/taxonomy/88869", headers=HEADERS)
data = resp.json()

print("status          :", data["status"])
print("scientificName  :", data["scientificName"])
print()
print("accepted block  :")
for k, v in data["accepted"].items():
    print(f"  {k}: {v}")

status          : synonym
scientificName  : Agaricus dryophilus

accepted block  :
  tid: 183886
  scientificName: Gymnopus dryophilus
  scientificNameAuthorship: (Bull.) Murrill
  taxonomicSource: None
  unacceptabilityReason: None
  taxonRemarks: None


So when we query a synonym, we can follow the chain:

```
"Agaricus dryophilus"  →  tid 88869  →  status=synonym  →  accepted tid 183886  →  "Gymnopus dryophilus"
```

The accepted block has **`tid`** and **`scientificName`** — that's all we need.  
The `unacceptabilityReason` and `taxonRemarks` fields are usually null.

---
## 3. `taxa/index.php?tid={tid}` — HTML synonym scraping

The REST API (`/api/v2/taxonomy/{tid}`) does **not** return the list of synonyms for an accepted taxon — it only tells you *whether* a given taxon is a synonym, and of what.

To get all synonyms, we have to fetch the HTML taxon page and parse the `synonymDiv` element.  
Here's what that raw HTML looks like before we parse it.

In [7]:
resp = requests.get(
    f"{BASE_URL}/taxa/index.php",
    params={"tid": 183886},
    headers=HEADERS,
)

syn_match = re.search(r'id="synonymDiv"[^>]*>(.*?)</div>', resp.text, re.DOTALL)
print(syn_match.group(1))

[<i>Agaricus dryophilus</i> Bull., <span class="synSpan"><a href="#" onclick="toggle('synSpan')" title="Click here to show more synonyms">more</a></span><span class="synSpan" onclick="toggle('synSpan')" style="display:none"><i>Collybia aquosa var. dryophila</i> (Bull.) Krieglst., <i>Collybia dryophila</i> (Bull.) P. Kumm., <i>Collybia dryophila var. alvearis</i> Cooke, <i>Collybia dryophila var. aurata</i> Quél., <i>Collybia dryophila var. oedipoides</i> Singer, <i>Collybia dryophila var. steinmannii</i> Raithelh., <i>Marasmius dryophilus</i> (Bull.) P. Karst., <i>Marasmius dryophilus var. alvearis</i> (Cooke) Rea, <i>Marasmius dryophilus var. auratus</i> (Quél.) Rea, <i>Omphalia dryophila</i> (Bull.) Gray</span>]


A few things to notice:

- All names are wrapped in `<i>` tags regardless of rank
- The portal hides most synonyms behind a JavaScript "more" toggle — but the HTML is all present in the DOM (in the hidden `<span class="synSpan">` block), so we get them all in one request without needing JavaScript execution
- Some names are infraspecific (contain `var.`, `subsp.`, etc.) — we filter those out because they're not species-level synonyms

In [8]:
# Extract all <i> names and split into species-level vs. infraspecific
INFRASPECIFIC_RE = re.compile(
    r"\b(var\.|subsp\.|ssp\.|f\.|fo\.|subf\.|cv\.|sect\.|subsect\.|ser\.)",
    re.IGNORECASE,
)

all_names = re.findall(r"<i>(.*?)</i>", syn_match.group(1))

species_level = [n for n in all_names if not INFRASPECIFIC_RE.search(n)]
infraspecific = [n for n in all_names if INFRASPECIFIC_RE.search(n)]

print("Species-level synonyms (kept):")
for n in species_level:
    print(f"  {n}")

print()
print("Infraspecific names (filtered out):")
for n in infraspecific:
    print(f"  {n}")

Species-level synonyms (kept):
  Agaricus dryophilus
  Collybia dryophila
  Marasmius dryophilus
  Omphalia dryophila

Infraspecific names (filtered out):
  Collybia aquosa var. dryophila
  Collybia dryophila var. alvearis
  Collybia dryophila var. aurata
  Collybia dryophila var. oedipoides
  Collybia dryophila var. steinmannii
  Marasmius dryophilus var. alvearis
  Marasmius dryophilus var. auratus


---
## 4. `/api/v2/occurrence/search` — Specimen records

This is a proper paginated REST endpoint that returns herbarium/collection records (specimens) for a given species.  
Each record is a Darwin Core–style observation or preserved specimen.

Useful query parameters:

| Parameter | Example | Meaning |
|---|---|---|
| `sciname` | `Gymnopus dryophilus` | Species name to search |
| `limit` | `10` | Records per page (default varies) |
| `offset` | `0` | Pagination offset |
| `country` | `Canada` | Filter by country |
| `stateProvince` | `British Columbia` | Filter by state/province |

In [9]:
resp = requests.get(
    f"{BASE_URL}/api/v2/occurrence/search",
    params={"sciname": "Gymnopus dryophilus", "limit": 3},
    headers=HEADERS,
)
data = resp.json()

print(f"Total records in portal : {data['count']}")
print(f"endOfRecords            : {data['endOfRecords']}")
print(f"Records returned        : {len(data['results'])}")

Total records in portal : 9632
endOfRecords            : False
Records returned        : 3


In [10]:
# Show the fields most likely to be useful for our project
FIELDS = [
    "occid", "sciname", "family",
    "recordedBy", "eventDate",
    "country", "stateProvince", "county", "locality",
    "decimalLatitude", "decimalLongitude",
    "basisOfRecord", "institutionCode",
]

for i, rec in enumerate(data["results"], 1):
    print(f"--- Record {i} ---")
    for f in FIELDS:
        print(f"  {f}: {rec.get(f)}")
    print()

--- Record 1 ---
  occid: 46
  sciname: Gymnopus dryophilus
  family: Marasmiaceae
  recordedBy: Mary Wells
  eventDate: 1966-08-07
  country: United States of America
  stateProvince: Colorado
  county: Gunnison 
  locality: Gunnison National Forest, Gold Creek, Ohio City
  decimalLatitude: 38.6501
  decimalLongitude: -106.5623
  basisOfRecord: PreservedSpecimen
  institutionCode: DBG

--- Record 2 ---
  occid: 2322
  sciname: Gymnopus dryophilus
  family: Marasmiaceae
  recordedBy: Vera S. Evenson
  eventDate: 1988-07-18
  country: United States of America
  stateProvince: Colorado
  county: Jefferson 
  locality: Coal Creek Canyon, Twin Spuce Road,
  decimalLatitude: 39.892244
  decimalLongitude: -105.36143
  basisOfRecord: PreservedSpecimen
  institutionCode: DBG

--- Record 3 ---
  occid: 4012
  sciname: Gymnopus dryophilus
  family: Marasmiaceae
  recordedBy: Vera S. Evenson
  eventDate: 1995-07-02
  country: United States of America
  stateProvince: Colorado
  county: Boulder 
 

In [11]:
# Filter to British Columbia records only
resp = requests.get(
    f"{BASE_URL}/api/v2/occurrence/search",
    params={
        "sciname": "Gymnopus dryophilus",
        "country": "Canada",
        "stateProvince": "British Columbia",
        "limit": 5,
    },
    headers=HEADERS,
)
data = resp.json()

print(f"BC records: {data['count']}")
for rec in data["results"]:
    print(f"  {rec['eventDate']}  {rec['locality']}  ({rec['decimalLatitude']}, {rec['decimalLongitude']})")

BC records: 56
  2000-06-05  Vancouver, Downtown Vancouver: Stanley Park  (49.30277778, -123.1444444)
  1940-06-29  Dog Creek Trail, 5 miles west of Fort Saint James Road, Cariboo Division.  (None, None)
  1940-06-19  Fort Saint James, Cariboo Division.  (None, None)
  1940-06-19  Fort Saint James, Cariboo Division.  (None, None)
  1940-06-22  Fort Saint James, Cariboo Division.  (None, None)
